# Adaptive Semantic JSCC — Stage 2 Colab Notebook

**Run on Google Colab with T4 GPU** (Runtime → Change runtime type → T4 GPU)

This notebook:
1. Uploads / refreshes the project zip
2. Installs requirements
3. Trains the ResNet-18 classifier on clean EuroSAT (~10-15 min)
4. Re-runs Model 1 eval WITH accuracy (now that classifier exists)
5. Smoke-tests Model 2 architecture
6. Trains Model 2 (~45-60 min)
7. Evaluates Model 2 at all SNRs and channels
8. Prints a side-by-side comparison table (Model 1 vs Model 2)

**Stage gate:** Model 2 PSNR is expected to be LOWER than Model 1.
This is not a bug — it sets up the SwinIR improvement in Stage 3.

In [ ]:
# ── CELL 1: Upload project zip and extract ───────────────────────────────────
# Upload your updated project zip (with classifier.py, models/resnet_jscc.py, etc.)
# then run this cell.

from google.colab import files
import zipfile, os, shutil

WORK_DIR = '/content/data_transmission'

uploaded = files.upload()   # select your zip file
zip_name = list(uploaded.keys())[0]
print(f'Uploaded: {zip_name}')

# Extract into WORK_DIR (overwrite existing files)
if os.path.exists(WORK_DIR):
    shutil.rmtree(WORK_DIR)
os.makedirs(WORK_DIR)
with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall(WORK_DIR)

%cd {WORK_DIR}
!ls -la

In [ ]:
# ── CELL 2: Verify GPU and install requirements ──────────────────────────────
import torch
print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU     : {torch.cuda.get_device_name(0)}')

!pip install -q -r requirements.txt
print('Requirements installed ✓')

In [ ]:
# ── CELL 3: Copy Model 1 checkpoint from Google Drive (if saved there) ───────
# Skip this cell if you already have the checkpoint in the project folder.
# Otherwise, mount Drive and copy the best.pt from your Stage 1 run.

from google.colab import drive
drive.mount('/content/drive')

import shutil, os
os.makedirs('checkpoints/model1', exist_ok=True)

# UPDATE this path to where you saved the Stage 1 checkpoint on Drive:
DRIVE_CKPT = '/content/drive/MyDrive/jscc/checkpoints/model1/best.pt'
shutil.copy(DRIVE_CKPT, 'checkpoints/model1/best.pt')
print('Model 1 checkpoint copied ✓')

In [ ]:
# ── CELL 4: Smoke-test Model 2 architecture ──────────────────────────────────
!python models/resnet_jscc.py

In [ ]:
# ── CELL 5: Train the ResNet-18 classifier on clean EuroSAT ──────────────────
# 30 epochs, ~10-15 minutes on T4.
# Expected clean test accuracy: ~95-98%.
# This classifier is ONLY used at eval time (never during JSCC training).
!python classifier.py --train

In [ ]:
# ── CELL 6: Re-run Model 1 evaluation WITH accuracy ─────────────────────────
# Now that the classifier exists, Model 1 gets its accuracy column.
# PSNR will be slightly different from Stage 1 because of the corrected
# averaging (dataset-level MSE, not batch-averaged MSE).
# Report the new numbers in your paper — they are more accurate.

!python eval.py \
    --config configs/model1.yaml \
    --checkpoint checkpoints/model1/best.pt \
    --clf_ckpt checkpoints/classifier/best.pt

In [ ]:
# ── CELL 7: Train Model 2 ────────────────────────────────────────────────────
# ResNet-18 (frozen) + JSCC MLP encoder + Channel + JSCC MLP decoder + CNN decoder
# Expected time: ~45-60 minutes on T4
# The backbone is frozen so only ~2M parameters are trained (vs ~5M in Model 1)
!python train.py --config configs/model2.yaml

In [ ]:
# ── CELL 8: Evaluate Model 2 on test set ────────────────────────────────────
!python eval.py \
    --config configs/model2.yaml \
    --checkpoint checkpoints/model2/best.pt \
    --clf_ckpt checkpoints/classifier/best.pt

In [ ]:
# ── CELL 9: Side-by-side comparison table ───────────────────────────────────
import pandas as pd
import os

m1_csv = 'results/model1/test_metrics.csv'
m2_csv = 'results/model2/test_metrics.csv'

m1 = pd.read_csv(m1_csv) if os.path.exists(m1_csv) else None
m2 = pd.read_csv(m2_csv) if os.path.exists(m2_csv) else None

for ch in ['awgn', 'rayleigh']:
    print(f'\n{"─"*70}')
    print(f'  Channel: {ch.upper()}')
    print(f'{"─"*70}')
    header = f'{"SNR":>6}  {"M1 PSNR":>10}  {"M2 PSNR":>10}  '\
             f'{"M1 SSIM":>9}  {"M2 SSIM":>9}  '\
             f'{"M1 Acc":>8}  {"M2 Acc":>8}'
    print(header)
    print('─' * 70)
    for snr in [0, 5, 10, 15, 20]:
        def get(df, col):
            if df is None:
                return 'N/A'
            row = df[(df['channel'] == ch) & (df['snr_db'] == snr)]
            if row.empty or pd.isna(row[col].values[0]):
                return 'N/A'
            v = row[col].values[0]
            return f'{v:.4f}' if col != 'accuracy' else f'{v:.4f}'
        print(f'{snr:>6}  '
              f'{get(m1,"psnr"):>10}  {get(m2,"psnr"):>10}  '
              f'{get(m1,"ssim"):>9}  {get(m2,"ssim"):>9}  '
              f'{get(m1,"accuracy"):>8}  {get(m2,"accuracy"):>8}')

print()
print('NOTE: Model 2 PSNR being LOWER than Model 1 is expected.')
print('Layer-4 of ResNet-18 on 64x64 is only 512x2x2 = 2048 values.')
print('This bottleneck is why Stage 3 (SwinIR) is needed.')

In [ ]:
# ── CELL 10: Save results to Drive ───────────────────────────────────────────
import shutil, os

# Mount Drive if not already mounted
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
except Exception:
    pass

DRIVE_BASE = '/content/drive/MyDrive/jscc'

for src, dst_name in [
    ('checkpoints/classifier',    'checkpoints/classifier'),
    ('checkpoints/model2',        'checkpoints/model2'),
    ('results/model1',            'results/model1_reeval'),   # updated with accuracy
    ('results/model2',            'results/model2'),
    ('results/classifier',        'results/classifier'),
]:
    dst = os.path.join(DRIVE_BASE, dst_name)
    if os.path.exists(src):
        if os.path.exists(dst):
            shutil.rmtree(dst)
        shutil.copytree(src, dst)
        print(f'Saved: {dst}')
    else:
        print(f'[SKIP] {src} not found')

print('\nAll results backed up to Google Drive ✓')
print('Stage 2 COMPLETE. Proceed to Stage 3 (Model 3 + SwinIR).')